# Model Evaluation Notebook\nRun this notebook to test your CNN model's performance on a dataset, calculate metrics (Accuracy, F1, Precision, Recall), and plot graphs (ROC Curve, Confusion Matrix).\n

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'



In [ ]:
BASE_DIR = Path('..').resolve()
after_pca_path = BASE_DIR / 'data' / 'raw' / 'after_pca.txt'
hla_path = BASE_DIR / 'data' / 'raw' / 'ParatopeIMGTopsi.txt'
model_weights_path = BASE_DIR / 'models' / 'CNN_WEIGHT_OPSI'

# Load required data for preprocessing
after_pca = np.loadtxt(after_pca_path)
hla = pd.read_csv(hla_path, sep='\t')
hla_dic = HLA_Dictionary(hla)
inventory = list(hla_dic.keys())
dic_inventory = dictionary(inventory)

# Load model
cnn_model = arsitekturCNN()
cnn_model.load_weights(str(model_weights_path) + '/')
print("Model loaded successfully!")\n

In [ ]:
# Load your dataset here for evaluation
# For example, using the IEDB Dataset
dataset_path = BASE_DIR / 'data' / 'raw' / 'IEDB Dataset.csv'
df = pd.read_csv(dataset_path)

print("Original Data:")
display(df.head())

# Preprocess the ground truth labels
if 'immunogenicity' in df.columns:
    if df['immunogenicity'].dtype == object:
        # Convert 'Positive' to 1, others to 0
        df['label'] = df['immunogenicity'].apply(lambda x: 1 if str(x).strip().lower() == 'positive' else 0)
    else:
        # Assuming they are already numerical probabilities, threshold at 0.5
        df['label'] = df['immunogenicity'].apply(lambda x: 1 if float(x) >= 0.5 else 0)
else:
    print("Warning: 'immunogenicity' column not found. You need labels for evaluation.")

# Drop rows with missing peptide or HLA
df = df.dropna(subset=['peptide', 'HLA'])
print(f"\nTotal samples for evaluation: {len(df)}")\n

In [ ]:
# The construct_aaindex function expects a DataFrame with 'peptide', 'HLA', and 'immunogenicity'
eval_df = df[['peptide', 'HLA']].copy()
eval_df['immunogenicity'] = ['0'] * len(eval_df)  # Dummy target for preprocessing

# Construct inputs for the model
dataset_score = construct_aaindex(eval_df, hla_dic, after_pca, dic_inventory)

input1_score = peptide_iterate(dataset_score)
input2_score = hla_iterate(dataset_score)

print("Running predictions...")
predictions_prob = cnn_model.predict(x=[input1_score, input2_score], verbose=1)
df['pred_prob'] = predictions_prob

# Convert probability to binary prediction (threshold = 0.5)
threshold = 0.5
df['pred_label'] = (df['pred_prob'] > threshold).astype(int)

display(df[['peptide', 'HLA', 'label', 'pred_prob', 'pred_label']].head())\n

In [ ]:
y_true = df['label'].values
y_pred = df['pred_label'].values
y_prob = df['pred_prob'].values

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

try:
    auc = roc_auc_score(y_true, y_prob)
except ValueError:
    auc = np.nan # If only one class is present in true labels

print("=== Evaluation Metrics ===")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")\n

In [ ]:
# 1. Plot ROC Curve
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
if not np.isnan(auc):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {auc:.3f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
else:
    plt.text(0.5, 0.5, 'Need both positive and negative labels\nfor ROC curve', 
             horizontalalignment='center', verticalalignment='center')
    plt.title('ROC Curve')

# 2. Plot Confusion Matrix
plt.subplot(1, 2, 2)
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Negative (0)', 'Positive (1)'],
            yticklabels=['Negative (0)', 'Positive (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')

plt.tight_layout()
plt.show()\n